In [6]:
# PSNR과 SSIM을 1초 만에 계산해 주는 만능 라이브러리 설치!
!pip install torchmetrics

!pip install compressai

In [7]:
import math
import torch
import torch.nn as nn
from torchvision import transforms
from compressai.zoo import bmshj2018_hyperprior
from torchmetrics.functional.image import peak_signal_noise_ratio as psnr
from torchmetrics.functional.image import structural_similarity_index_measure as ssim

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
def evaluate_metrics(model, data_loader, device='cuda'):
    """
    모든 데이터에 대해 모델을 평가하고 BPP, PSNR, SSIM 평균을 반환합니다.
    """
    model.eval() # 무조건 평가 모드로 변경 (학습 안 됨!)
    model.update(force=True) # 압축기 초기화 세팅

    total_bpp = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    count = 0

    print("📏 자동 채점관: 모델 성능 평가를 시작합니다...")

    with torch.no_grad():
        for low_img, high_img in data_loader:
            low_img = low_img.to(device)
            high_img = high_img.to(device)

            # [Step 1] 진짜 0과 1 바이너리(Bytes)로 압축해서 '용량(BPP)' 구하기
            compressed_data = model.compress(low_img)

            # 스트링(바이트)들의 길이 다 더하기
            strings_y = compressed_data['strings'][0]
            strings_z = compressed_data['strings'][1]
            total_bytes = sum(len(s) for s in strings_y) + sum(len(s) for s in strings_z)

            # 현재 이미지들의 전체 픽셀 수 계산 (N * H * W)
            num_pixels = low_img.size(0) * low_img.size(2) * low_img.size(3)

            # 용량 점수: 비트를 픽셀로 나누기 (작을수록 좋음!)
            bpp = (total_bytes * 8) / num_pixels
            total_bpp += bpp

            # [Step 2] 압축된 바이너리를 다시 '이미지로 복원'하기
            decompressed_data = model.decompress(compressed_data['strings'], compressed_data['shape'])
            x_hat = decompressed_data['x_hat'].clamp(0, 1) # 안전하게 0~1사이 고정

            # [Step 3] 화질(PSNR & SSIM) 구하기! (복원된 이미지 vs 진짜 밝은 정답지)
            batch_psnr = psnr(x_hat, high_img, data_range=1.0)
            batch_ssim = ssim(x_hat, high_img, data_range=1.0)

            total_psnr += batch_psnr.item()
            total_ssim += batch_ssim.item()
            count += 1

    # 평균 계산
    avg_bpp = total_bpp / count
    avg_psnr = total_psnr / count
    avg_ssim = total_ssim / count

    print(f"✅ 평가 완료! (Total batches: {count})")
    print("-" * 30)
    print(f"📊 BPP (용량) : {avg_bpp:.4f}  [👇낮을수록 가볍고 통신비 적게 나옴]")
    print(f"📺 PSNR(픽셀 화질): {avg_psnr:.4f} dB [👆높을수록 정답 사진과 색깔/위치가 일치함]")
    print(f"👁️ SSIM(인간 인지): {avg_ssim:.4f}     [👆1.0에 가까울수록 눈에 보기에 구조가 완벽함]")
    print("-" * 30)

    return avg_bpp, avg_psnr, avg_ssim

print("🎉 평가(Evaluation) 함수 정의 완료!")

🎉 평가(Evaluation) 함수 정의 완료!


# Baseline 1으로 테스트 진행

In [12]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from compressai.zoo import bmshj2018_hyperprior

# ----------------------------------------------------
# 1. 아까 만들었던 그 데이터셋 클래스를 평가용으로 똑같이 가져옵니다!
# ----------------------------------------------------
class LOLDataset(Dataset):
    def __init__(self, root_dir, crop_size=256, is_train=False):
        self.low_dir = os.path.join(root_dir, 'low')
        self.high_dir = os.path.join(root_dir, 'high')
        self.image_names = sorted(os.listdir(self.low_dir))
        self.crop_size = crop_size
        self.is_train = is_train

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        low_img = Image.open(os.path.join(self.low_dir, img_name)).convert('RGB')
        high_img = Image.open(os.path.join(self.high_dir, img_name)).convert('RGB')

        w, h = TF.get_image_size(low_img)

        # [중요] 평가(Test)할 때는 이미지를 랜덤하게 자르지 않고 항상 가운데(Center)를 정직하게 자릅니다!
        if self.is_train:
            crop_i = random.randint(0, h - self.crop_size)
            crop_j = random.randint(0, w - self.crop_size)
            if random.random() > 0.5:
                low_img = TF.hflip(low_img)
                high_img = TF.hflip(high_img)
        else:
            # 중앙 정렬 크롭
            crop_i = (h - self.crop_size) // 2
            crop_j = (w - self.crop_size) // 2

        low_img = TF.crop(low_img, crop_i, crop_j, self.crop_size, self.crop_size)
        high_img = TF.crop(high_img, crop_i, crop_j, self.crop_size, self.crop_size)

        low_tensor = TF.to_tensor(low_img)
        high_tensor = TF.to_tensor(high_img)

        return low_tensor, high_tensor

# ----------------------------------------------------
# 2. 시험지(Test Data) 불러오기 준비!
# ----------------------------------------------------
# ★ 경로 주의! 'our485'가 아니라 평가용 사진 15장이 있는 'eval15' 폴더로 연결합니다!
# 본인의 드라이브 경로로 맞춰주세요!
eval_path = '/content/drive/MyDrive/CUAI_summer_conference/LOL_Dataset/lol_dataset/eval15'

# 평가 모드이므로 is_train=False를 줘서 중앙을 딱 맞게 자릅니다.
test_dataset = LOLDataset(root_dir=eval_path, crop_size=256, is_train=False)

# 시험을 치는 테스트 로더 완성! (평가니까 순서를 안 섞어도 됩니다 shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
print("✅ 시험용 test_loader 생성 완료! (15장 묶음)")


# ----------------------------------------------------
# 3. 진짜 성능 평가해보기 ( Baseline 1 테스트 )
# ----------------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 튜토리얼 B1 모델 불러오기 (학습 안 시킨 순정품)
print("\n[Baseline 1 - Quality=2 모델을 소환합니다...]")
model_B1_Q2 = bmshj2018_hyperprior(quality=2, pretrained=True).to(device)

# 아까 만들어둔 자동채점관 함수 evaluate_metrics()가 선언되어 있다고 가정합니다!
# 여기에 방금 만든 test_loader를 쑥 집어넣으면 됩니다!
b1_bpp, b1_psnr, b1_ssim = evaluate_metrics(model=model_B1_Q2, data_loader=test_loader, device=device)

✅ 시험용 test_loader 생성 완료! (15장 묶음)

[Baseline 1 - Quality=2 모델을 소환합니다...]
📏 자동 채점관: 모델 성능 평가를 시작합니다...
✅ 평가 완료! (Total batches: 4)
------------------------------
📊 BPP (용량) : 0.0493  [👇낮을수록 가볍고 통신비 적게 나옴]
📺 PSNR(픽셀 화질): 7.2592 dB [👆높을수록 정답 사진과 색깔/위치가 일치함]
👁️ SSIM(인간 인지): 0.1804     [👆1.0에 가까울수록 눈에 보기에 구조가 완벽함]
------------------------------


PSNR: 완전히 수학적인 오차입니다. 윤곽선(엣지)이 조금 어긋났을 뿐 눈으로 보기엔 별 차이가 없어도, 점수는 나락을 가는 아주 피도 눈물도 없는(Oversmoothing과 픽셀 어긋남에 매우 엄격한) 수학 점수입니다. 30dB 정도만 넘으면 매우 화질이 우수하다는 뜻입니다. (Baseline 1은 20dB대 초반 이하의 낮은 숫자가 나올 것입니다!)


SSIM: 인간의 눈에 맞춘 인지 과학적 점수입니다. "이 모델이 얼마나 사진 전체 구조(윤곽 덩어리, 대비, 조명 톤)를 정답과 비슷하게 뽑아냈는가?"를 평가합니다. 우리가 추후에 도입할 Gaussian Smoothed Edge Loss 나 Perceptual Loss 는 이 SSIM 점수를 비약적으로 올리는 것을 목표로 합니다.

---
# Baseline 3 결과 확인 (체크포인트 불러와서 실제 사진 비교)

**목적:** vast.ai 인스턴스는 지웠지만, 학습된 체크포인트가 Hugging Face에 백업돼 있으니
**Colab에서 그 파일만 불러와서** 실제 복원 사진과 PSNR/SSIM을 확인합니다.

이 셀들은 위 Baseline 1 셀과 독립적으로 실행 가능합니다 (이 섹션부터 실행해도 됨).

In [ ]:
# 0. 필요한 패키지 설치 (위에서 이미 설치했으면 건너뛰어도 됨)
!pip install -q compressai torchmetrics huggingface_hub

In [ ]:
# 1. Hugging Face 로그인 (private 레포 접근용)
from huggingface_hub import login, hf_hub_download, snapshot_download
login()  # 토큰 붙여넣기 (huggingface.co/settings/tokens 에서 Read 토큰이면 충분)

In [ ]:
# 2. 체크포인트 다운로드 (본인 개인 백업 레포에서)
ckpt_path = hf_hub_download(
    repo_id="snnipe/baseline3-checkpoints",
    filename="baseline3_FINAL_none_Q2_Ratio0.pth",  # 실제 파일명이 다르면 여기 수정
    repo_type="model",
)
print("체크포인트 경로:", ckpt_path)

In [ ]:
# 3. 모델 구조 재구성 (학습 때와 완전히 동일하게: CBAM 없음, quality=2)
import torch
from compressai.zoo import bmshj2018_hyperprior

device = "cuda" if torch.cuda.is_available() else "cpu"

model = bmshj2018_hyperprior(quality=2, pretrained=False).to(device)
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
model.update(force=True)  # 압축 엔트로피 테이블 초기화 (평가 전 필수)
print("✅ Baseline 3 체크포인트 로드 완료")

In [ ]:
# 4. eval15 데이터셋 받기 (평가용 15장, 학습에 안 쓰인 데이터)
data_dir = snapshot_download(
    repo_id="CUAI-CV-Team4/lol_dataset",
    repo_type="dataset",
    allow_patterns=["lol_dataset/eval15/*"],  # zip이 아니라 이미 풀린 구조라면 이 패턴 조정 필요
)
print("데이터 경로:", data_dir)

# ⚠️ 만약 데이터셋이 archive.zip 하나로만 돼 있다면 아래처럼 받아서 직접 풀어야 함:
# zip_path = hf_hub_download(repo_id="CUAI-CV-Team4/lol_dataset", filename="archive.zip", repo_type="dataset")
# import zipfile; zipfile.ZipFile(zip_path).extractall("/content/lol_data")
# data_dir = "/content/lol_data"

In [ ]:
# 5. 저조도(low) 사진 몇 장 불러와서 실제 복원 시켜보기
import os, glob
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import math

eval_root = os.path.join(data_dir, "lol_dataset", "eval15")
low_paths = sorted(glob.glob(os.path.join(eval_root, "low", "*.png")))[:4]

def load_img(p, crop=None):
    img = Image.open(p).convert("RGB")
    t = TF.to_tensor(img).unsqueeze(0)
    return t

def psnr_val(a, b):
    mse = torch.mean((a - b) ** 2).item()
    return 100.0 if mse == 0 else 10 * math.log10(1.0 / mse)

fig, axes = plt.subplots(len(low_paths), 3, figsize=(12, 4 * len(low_paths)))
for i, low_path in enumerate(low_paths):
    name = os.path.basename(low_path)
    high_path = low_path.replace("/low/", "/high/")

    low = load_img(low_path).to(device)
    high = load_img(high_path).to(device)

    # bmshj2018은 H,W가 64의 배수여야 함 -> 중앙 crop
    _, _, H, W = low.shape
    H2, W2 = (H // 64) * 64, (W // 64) * 64
    top, left = (H - H2) // 2, (W - W2) // 2
    low = low[:, :, top:top+H2, left:left+W2]
    high = high[:, :, top:top+H2, left:left+W2]

    with torch.no_grad():
        out = model(low)
        x_hat = out["x_hat"].clamp(0, 1)

    p = psnr_val(x_hat, high)

    axes[i, 0].imshow(low[0].cpu().permute(1, 2, 0)); axes[i, 0].set_title(f"1. Low (Input)\n{name}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(x_hat[0].cpu().permute(1, 2, 0)); axes[i, 1].set_title(f"2. Baseline3 복원 (PSNR {p:.2f}dB)"); axes[i, 1].axis("off")
    axes[i, 2].imshow(high[0].cpu().permute(1, 2, 0)); axes[i, 2].set_title("3. High (정답)"); axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

print("👉 2번(복원) 열에서 간판 글씨나 윤곽선이 흐릿하게 뭉개져 보이면, Baseline 3의 Oversmoothing 문제가 확인된 것입니다.")